[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulrobustelli/CSMP_150_2026/blob/main/notebooks/maxent_die.ipynb)

# Maximum entropy: a loaded die

Change the target mean to solve for the maximum-entropy probabilities, or change the Lagrange multiplier directly to explore its effect.

We maximize $S=-\sum_{i=1}^6p_i\ln p_i$ subject to $\sum_i p_i=1$ and $\sum_i i p_i=\mu$.

The solution is
$$p_i(\lambda)=\frac{e^{-\lambda i}}{Z(\lambda)},\qquad Z(\lambda)=\sum_{i=1}^6e^{-\lambda i}.$$

The solver finds the root of $\sum_i i p_i(\lambda)-\mu=0$ using Brent's method. This mean decreases monotonically with $\lambda$, since its derivative is $-\operatorname{Var}_p(i)$.

Natural logarithms give entropy in **nats**. We use $0\ln0=0$. Target means 1 and 6 require limiting distributions with infinite multipliers.


In [ ]:
# Enable interactive widgets in Google Colab.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.optimize import brentq
from IPython.display import display, clear_output

# Die faces are the states; their numerical values are the observable.
faces = np.arange(1, 7, dtype=float)

def probabilities(lam):
    """Stable evaluation of p_i = exp(-lambda*i) / Z."""
    log_weights = -lam * faces
    weights = np.exp(log_weights - log_weights.max())
    return weights / weights.sum()

def solve_mean(target):
    """The endpoints are limiting distributions (infinite lambda)."""
    if target == 1:
        return np.inf, np.array([1., 0, 0, 0, 0, 0])
    if target == 6:
        return -np.inf, np.array([0., 0, 0, 0, 0, 1])
    if target == 3.5:
        return 0., np.full(6, 1/6)
    bound = 1.
    residual = lambda lam: faces @ probabilities(lam) - target
    while residual(-bound) < 0 or residual(bound) > 0:
        bound *= 2
    lam = brentq(residual, -bound, bound, xtol=1e-13)
    return lam, probabilities(lam)

mode = widgets.ToggleButtons(
    options=['Solve for target mean', 'Explore lambda'],
    description='Mode:', style={'description_width': 'initial'})
mean_slider = widgets.FloatSlider(value=3, min=1, max=6, step=.01,
    description='Target mean:', readout_format='.2f', continuous_update=False,
    style={'description_width': 'initial'}, layout=widgets.Layout(width='95%'))
lambda_slider = widgets.FloatSlider(value=.174629, min=-5, max=5, step=.01,
    description='Lambda:', readout_format='.3f', continuous_update=False,
    style={'description_width': 'initial'}, layout=widgets.Layout(width='95%'))
output = widgets.Output()

def update(change=None):
    solving = mode.value == 'Solve for target mean'
    mean_slider.layout.display = '' if solving else 'none'
    lambda_slider.layout.display = 'none' if solving else ''
    if solving:
        lam, p = solve_mean(mean_slider.value)
    else:
        lam = lambda_slider.value
        p = probabilities(lam)
    mean = faces @ p
    positive = p > 0
    entropy = -np.sum(p[positive] * np.log(p[positive]))
    with output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4.2))
        ax.bar(faces, p, width=.7, color='#188579')
        for x, value in zip(faces, p):
            ax.text(x, value + .025, f'{value:.3f}', ha='center', fontsize=12)
        ax.set(xlabel='Die outcome', ylabel='Probability', xticks=faces,
               ylim=(0, 1.12), xlim=(.4, 6.6))
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_title('Maximum-entropy die distribution', fontsize=16, pad=16)
        fig.tight_layout()
        plt.show()
        lam_text = f'{lam:.6f}' if np.isfinite(lam) else ('+infinity (limit)' if lam > 0 else '-infinity (limit)')
        print(f'Mean = {mean:.6f}    Lambda = {lam_text}    Entropy = {entropy:.6f} nats')
        print('Probabilities:', np.array2string(p, precision=5))

for control in (mode, mean_slider, lambda_slider):
    control.observe(update, names='value')
display(widgets.VBox([mode, mean_slider, lambda_slider, output]))
update()


## Explore

1. Set the target mean to 3.5. Why is the distribution uniform? What is its entropy?
2. Set the mean to 3. Compare the resulting probabilities with the lecture example.
3. Compare means 2 and 5. How are their distributions and entropies related?
4. Switch to exploring lambda. What do positive and negative values favor?
5. Approach means 1 and 6. What happens to the entropy and multiplier?

## From a die to a molecular ensemble

Replace each face with a conformation and each face value with a predicted observable. With nonuniform prior weights $q_i$, the update becomes $p_i\propto q_i e^{-\lambda f(x_i)}$. Multiple observables require a vector of multipliers.
